In [1]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
! pip install datasets

In [3]:
from datasets import load_dataset
dataset =  load_dataset("bentrevett/multi30k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
train,val,test = dataset["train"],dataset["validation"],dataset["test"]

## Building the tokenization

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import tqdm

In [6]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [7]:
train[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

In [8]:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 109.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
## loading the tokenizer
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

In [10]:
sentence = "hi how are you"
[text.text for text in en_nlp.tokenizer(sentence)]

['hi', 'how', 'are', 'you']

In [11]:
def tokenize_examples(example,en_nlp,de_nlp,lower,sos_token,eos_token,max_length):
  english = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
  german = [token.text  for token in de_nlp.tokenizer(example["de"])][:max_length]
  if lower:
    english = [text.lower() for text in english]
    german = [text.lower() for text in german]
  english = [sos_token] + english + [eos_token]
  german = [sos_token] + german + [eos_token]
  return {"en_tokens":english,"de_tokens":german}

In [12]:
tokenize_examples(train[0],en_nlp,de_nlp,1,"<sos>","<eos>",100)

{'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

In [13]:
max_length = 128
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
    "max_length": max_length
}

In [14]:
train_data = train.map(tokenize_examples, fn_kwargs=fn_kwargs)
valid_data = val.map(tokenize_examples, fn_kwargs=fn_kwargs)
test_data = test.map(tokenize_examples, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Vocabulary

In [15]:
## so lets build the vocabulary
unk_token = "<unk>" # to capture the unknown token while inference
pad_token = "<pad>" # to make the len same
special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

In [16]:
train_data["en_tokens"][0]

['<sos>',
 'two',
 'young',
 ',',
 'white',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.',
 '<eos>']

In [17]:
from collections import Counter
vocab = {
    "<unk>":0,
    "<pad>":1,
    "<sos>":2,
    "<eos>":3
}

In [18]:
Counter

collections.Counter

In [19]:
for sentence in train_data["en_tokens"]:
  for token in sentence:
    if token not in vocab:
      vocab[token] = len(vocab) + 1
  break

In [20]:
sentence

['<sos>',
 'two',
 'young',
 ',',
 'white',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.',
 '<eos>']

In [21]:
vocab

{'<unk>': 0,
 '<pad>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'two': 5,
 'young': 6,
 ',': 7,
 'white': 8,
 'males': 9,
 'are': 10,
 'outside': 11,
 'near': 12,
 'many': 13,
 'bushes': 14,
 '.': 15}

In [22]:
en_vocab =  {
    "<unk>":0,
    "<pad>":1,
    "<sos>":2,
    "<eos>":3
}
# english
en_counter = Counter()
for sentence in train_data["en_tokens"]:
    en_counter.update(sentence)
min_freq = 2
for token, count in en_counter.items():
    if count >= min_freq and token not in en_vocab:
        en_vocab[token] = len(en_vocab)

In [23]:
en_vocab

{'<unk>': 0,
 '<pad>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'two': 4,
 'young': 5,
 ',': 6,
 'white': 7,
 'males': 8,
 'are': 9,
 'outside': 10,
 'near': 11,
 'many': 12,
 'bushes': 13,
 '.': 14,
 'several': 15,
 'men': 16,
 'in': 17,
 'hard': 18,
 'hats': 19,
 'operating': 20,
 'a': 21,
 'giant': 22,
 'pulley': 23,
 'system': 24,
 'little': 25,
 'girl': 26,
 'climbing': 27,
 'into': 28,
 'wooden': 29,
 'playhouse': 30,
 'man': 31,
 'blue': 32,
 'shirt': 33,
 'is': 34,
 'standing': 35,
 'on': 36,
 'ladder': 37,
 'cleaning': 38,
 'window': 39,
 'at': 40,
 'the': 41,
 'stove': 42,
 'preparing': 43,
 'food': 44,
 'green': 45,
 'holds': 46,
 'guitar': 47,
 'while': 48,
 'other': 49,
 'observes': 50,
 'his': 51,
 'smiling': 52,
 'stuffed': 53,
 'lion': 54,
 'trendy': 55,
 'talking': 56,
 'her': 57,
 'cellphone': 58,
 'gliding': 59,
 'slowly': 60,
 'down': 61,
 'street': 62,
 'woman': 63,
 'with': 64,
 'large': 65,
 'purse': 66,
 'walking': 67,
 'by': 68,
 'gate': 69,
 'boys': 70,
 'dancing': 71,
 

In [24]:
de_vocab =  {
    "<unk>":0,
    "<pad>":1,
    "<sos>":2,
    "<eos>":3
}
# german
de_counter = Counter()
for sentence in train_data["de_tokens"]:
    de_counter.update(sentence)
min_freq = 2
for token, count in de_counter.items():
    if count >= min_freq and token not in de_vocab:
        de_vocab[token] = len(de_vocab)

In [25]:
en_vocab["the"]

41

In [26]:
len(en_vocab),len(de_vocab)

(5893, 7853)

In [27]:
# just and test
"The" in en_vocab

False

In [28]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]
# to check that they got the came index

In [29]:
unk_index , pad_index

(0, 1)

In [30]:
def lookup_indices(example,en_vocab,de_vocab):
  en_indices,de_indices = [],[]
  #english
  for token in example["en_tokens"]:
    if token not in en_vocab:
      en_indices.append(en_vocab["<unk>"])
    else:  # Add this else!
        en_indices.append(en_vocab[token])
  #german
  for token in example["de_tokens"]:
    if token not in de_vocab:
      de_indices.append(de_vocab["<unk>"])
    else:  # Add this else!
        de_indices.append(de_vocab[token])
  return {"en_ids":torch.tensor(en_indices),"de_ids":torch.tensor(de_indices)}

In [31]:
lookup_indices(train_data[0],en_vocab,de_vocab)

{'en_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  3]),
 'de_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,  3])}

In [32]:
train[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

In [33]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

In [34]:
fn_kwargs = {
    "en_vocab":en_vocab,
    "de_vocab":de_vocab
}
train_data = train_data.map(lookup_indices,fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

In [35]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3],
 'de_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]}

In [36]:
valid_data = valid_data.map(lookup_indices,fn_kwargs=fn_kwargs)
test_data = test_data.map(lookup_indices,fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [37]:
# reverse vocab
en_itos = {i: s for s, i in en_vocab.items()}
de_itos = {i: s for s, i in de_vocab.items()}

def lookup_vocab(example,en_itos,de_itos):
  en_indices,de_indices = [],[]
  #english
  for ids in example["en_ids"]:
    en_indices.append(en_itos[ids])
  #german
  for ids in example["de_ids"]:
    de_indices.append(de_itos[ids])
  return {"en_tokens_rev":(en_indices),"de_tokens_rev":(de_indices)}

In [38]:
train_data

Dataset({
    features: ['en', 'de', 'en_tokens', 'de_tokens', 'en_ids', 'de_ids'],
    num_rows: 29000
})

In [39]:
lookup_indices(train_data[0],en_vocab,de_vocab)

{'en_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  3]),
 'de_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,  3])}

In [40]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3],
 'de_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]}

## Data Loader

In [41]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [torch.tensor(example["en_ids"]) for example in batch]
        batch_de_ids = [torch.tensor(example["de_ids"]) for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index) # padding as the same length for one batch
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [42]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [43]:
batch_size = 32

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

In [44]:
for x in train_data_loader:
  break

In [45]:
len(x["en_ids"]) , len(x["de_ids"])

(26, 24)

## Next model training

In [46]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math

In [47]:
## Embedding
class inputEmbedding(nn.Module):
  def __init__(self,vocab_size,embedding_dim):
    super().__init__()
    self.vocab_size = vocab_size
    self.embedding_dim = embedding_dim
    self.embedding = nn.Embedding(vocab_size,embedding_dim)
  def forward(self,x):
    x = self.embedding(x)
    return x

In [65]:
## Positional Encoding
class positionalEncoding(nn.Module):
  def __init__(self,seq_len,d_k):
    super().__init__()
    self.seq_len = seq_len
    self.d_k = d_k
    pe = torch.zeros(seq_len,d_k)
    div = torch.exp(torch.arange(0,self.d_k,2).float() * - (math.log(10000.0)/self.d_k))
    positions = torch.arange(0,self.seq_len,dtype=torch.float).unsqueeze(1)
    pe[:,0::2] = torch.sin(positions*div)
    pe[:,1::2] = torch.cos(positions*div)
    pe = pe.unsqueeze(0)
    self.register_buffer("pe",pe)
  def forward(self,x):
    return x + self.pe[:,:x.size(1),:].requires_grad_(False)

In [49]:
## Feed Forward Network
class FeedForward(nn.Module):
  def __init__(self,d_k):
    super().__init__()
    self.d_k = d_k
    self.fnn = nn.Sequential(
        nn.Linear(self.d_k,2048),
        nn.ReLU(),
        nn.Linear(2048,self.d_k)
    )
  def forward(self,x):
    return self.fnn(x)

In [50]:
## layer norm
class layerNorm(nn.Module):
  def __init__(self,embedding_dim):
    super().__init__()
    self.alpha = nn.Parameter(torch.ones(embedding_dim))
    self.beta = nn.Parameter(torch.ones(embedding_dim))
    self.eps = 0.001
    pass
  def forward(self,x):
    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True)
    x = self.alpha * (x-mean)/(self.eps+std) + self.beta
    return x

In [51]:
## Multihead Self Attention
class MultiHeadAttention(nn.Module):
  def __init__(self,d_k,heads,head_dim,device):
    super().__init__()
    self.d_k = d_k
    assert d_k == heads*head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.Q = nn.Linear(self.d_k,self.d_k)
    self.K = nn.Linear(self.d_k,self.d_k)
    self.V = nn.Linear(self.d_k,self.d_k)
    self.device = device
    self.out = nn.Linear(self.d_k,self.d_k)
  def forward(self,x,mask):
    B,S,E = x.size()
    q = self.Q(x).view(B,S,self.heads,self.head_dim).transpose(1,2) # [32,100,128] => [32,100,4,32]
    k = self.K(x).view(B,S,self.heads,self.head_dim).transpose(1,2)
    v = self.V(x).view(B,S,self.heads,self.head_dim).transpose(1,2)
    k = k.transpose(-2,-1) # [32,4,100,32]
    attention_scores = ((q @ k)/math.sqrt(self.d_k))
    if mask :
      mat = torch.tril(torch.ones(S,S)).to(self.device)
      mat = mat.unsqueeze(0).unsqueeze(0)
      attention_scores = attention_scores.masked_fill(mat == 0,-1e9) ## where where their is 0 put to -infinity
    attention_scores = torch.softmax(attention_scores,dim=-1) @ v
    attention_score = attention_scores.transpose(1,2).contiguous().view(B,S,E)
    return self.out(attention_score)

In [82]:
## cross attention
class CrossAttention(nn.Module):
  def __init__(self,d_k,heads,head_dim):
    super().__init__()
    self.d_k = d_k
    assert d_k == heads*head_dim
    self.heads = heads
    self.head_dim = head_dim
    self.Q = nn.Linear(self.d_k,self.d_k)
    self.out = nn.Linear(self.d_k,self.d_k)
  def forward(self,x,k,v):
    B,S,E = x.size()
    B, S_enc, E_enc = k.size()
    q = self.Q(x).view(B,S,self.heads,self.head_dim).transpose(1,2)
    k = k.view(B,S_enc,self.heads,self.head_dim).transpose(1,2)
    v = v.view(B,S_enc,self.heads,self.head_dim).transpose(1,2)
    k = k.transpose(-2,-1) # [32,4,100,32]
    attention_scores = ((q @ k)/math.sqrt(self.d_k))
    attention_scores = torch.softmax(attention_scores,dim=-1) @ v
    attention_score = attention_scores.transpose(1,2).contiguous().view(B,S,E)
    return self.out(attention_score)

In [83]:
## residual layer
class residualLayer(nn.Module):
  def __init__(self):
    super().__init__()
    self.norm = layerNorm()
    self.dropout = nn.Dropout(0.1)
  def forward(self,x,sub_layer):
    return self.norm(x + self.dropout(sub_layer))

In [84]:
class EncoderBlock(nn.Module):
  def __init__(self,attention:MultiHeadAttention,fnn:FeedForward,d_k):
    super().__init__()
    self.attention = attention
    self.attention = attention
    self.fnn = fnn
    self.norm = layerNorm(d_k)
  def forward(self,x):
    # here the x coems is positionally encoded
    x = self.norm(x + self.attention(x,0))
    x = self.norm(x + self.fnn(x))
    return x

In [85]:
class DecoderBlock(nn.Module):
  def __init__(self,attention:MultiHeadAttention,fnn:FeedForward,cr:CrossAttention,d_k):
    super().__init__()
    self.attention = attention
    self.attention = attention
    self.fnn = fnn
    self.norm = layerNorm(d_k)
    self.cr = cr
    # self.k = nn.Linear(d_k,d_k)
    # self.V = nn.Linear(d_k,d_k)
  def forward(self,x,k,v):
    # here the x coems is positionally encoded
    x = self.norm(x + self.attention(x,1))
    x = self.norm(x + self.cr(x,k,v))
    x = self.norm(x + self.fnn(x))
    return x

In [86]:
class InitialLayer(nn.Module):
  def __init__(self,embedding:inputEmbedding,positional:positionalEncoding):
    super().__init__()
    self.embedding = embedding
    self.pos = positional
  def forward(self,x):
    x = self.pos(self.embedding(x))
    return x

In [87]:
class OutputLayer(nn.Module):
  def __init__(self,d_k,vocab_size):
    super().__init__()
    self.d_k = d_k
    self.out = nn.Linear(d_k,vocab_size)
  def forward(self,x):
    return self.out(x) # cross entropy as softmax

In [88]:
from torch.nn.modules import ModuleList
class Encoder(nn.Module):
  def __init__(self,sublist:nn.ModuleList,d_k):
    super().__init__()
    self.sublayer = sublist
    self.norm = layerNorm(d_k)
  def forward(self,x):
    for layer in self.sublayer:
      x = layer(x)
    return self.norm(x)

In [89]:
from torch.nn.modules import ModuleList
class Decoder(nn.Module):
  def __init__(self,sublist:nn.ModuleList,d_k):
    super().__init__()
    self.sublayer = sublist
    self.norm = layerNorm(d_k)
  def forward(self,x,k,v):
    for layer in self.sublayer:
      x = layer(x,k,v)
    return self.norm(x)

In [90]:
class Transformer(nn.Module):
  def __init__(self,encoder:Encoder,decoder:Decoder,initial:InitialLayer,out:OutputLayer,d_k):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.out = out
    self.initial = initial
    self.K = nn.Linear(d_k,d_k)
    self.V = nn.Linear(d_k,d_k)
  def forward(self,x,y):
    x = self.initial(x)
    y = self.initial(y)
    x = self.encoder(x)
    k,v = self.K(x),self.V(x)
    #print(k.shape , v.shape)
    y = self.decoder(y,k,v)
    y = self.out(y)
    return y

In [91]:
# --- Parameters ---
d_k = 128
heads = 4
head_dim = 32
v_size = len(de_vocab)
seq_len = max_length
device = "cuda" if torch.cuda.is_available() else "cpu"
# 1. Basic Components
emb = inputEmbedding(v_size, d_k)
pos = positionalEncoding(seq_len, d_k)
it = InitialLayer(emb, pos)
fnn = FeedForward(d_k)
att = MultiHeadAttention(d_k, heads, head_dim, device)
cr = CrossAttention(d_k, heads, head_dim)
out_layer = OutputLayer(d_k, v_size)

# 2. Build Stacks using ModuleList
# We create 6 unique layers for Encoder and Decoder
enc_blocks = nn.ModuleList([EncoderBlock(att, fnn, d_k) for _ in range(6)])
dec_blocks = nn.ModuleList([DecoderBlock(att, fnn, cr, d_k) for _ in range(6)])

encoder_stack = Encoder(enc_blocks, d_k)
decoder_stack = Decoder(dec_blocks, d_k)

# 3. Initialize Transformer
model = Transformer(encoder_stack, decoder_stack, it, out_layer, d_k).to(device)
# Expected: torch.Size([32, 1000, 5000])

In [92]:
len(en_vocab) , len(de_vocab)

(5893, 7853)

## Training transformer model for machine translation task

In [93]:
import torch.optim as optim
from torch.optim import lr_scheduler as schedular

## test part

In [94]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)
num_epochs = 1
for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for batch in train_data_loader:
        src = batch["en_ids"].to(device).transpose(0, 1) # Ensure [Batch, Seq] i.e [26,32] => [32,26]
        trg = batch["de_ids"].to(device).transpose(0, 1)

        optimizer.zero_grad()
        #print(len(src) , len(trg[:,:-1]))
        # Output: [batch, trg_len, vocab_size]
        output = model(src, trg[:, :-1]) # $[32, 23]$We remove the <eos> (last token) so the model predicts it.

        # Flatten for CrossEntropy
        output_dim = output.shape[-1] # the cross entropy expect the pred as the 2d matrix
        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1) # the cross entropy except the target as the 1d matrix

        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 32
32 3

In [96]:
epoch_loss / len(train_data_loader)

4.913483351567614

## test part over

In [99]:
import math
import time
from tqdm.notebook import tqdm

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    # Wrap validation loader with tqdm
    val_pbar = tqdm(iterator, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in val_pbar:
            src = batch["en_ids"].to(device).transpose(0, 1)
            trg = batch["de_ids"].to(device).transpose(0, 1)

            output = model(src, trg[:, :-1])

            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

            # Optional: Update progress bar with loss
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    return epoch_loss / len(iterator)

# Training Hyperparameters
num_epochs = 10
best_valid_loss = float('inf')

for epoch in range(num_epochs):
    start_time = time.time()

    # --- Training Phase ---
    model.train()
    train_loss = 0

    # Wrap training loader with tqdm
    train_pbar = tqdm(train_data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in train_pbar:
        src = batch["en_ids"].to(device).transpose(0, 1)
        trg = batch["de_ids"].to(device).transpose(0, 1)

        optimizer.zero_grad()
        output = model(src, trg[:, :-1])

        output_dim = output.shape[-1]
        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

        # Update progress bar with current batch loss
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    # --- Validation Phase ---
    avg_train_loss = train_loss / len(train_data_loader)
    avg_valid_loss = evaluate(model, valid_data_loader, criterion)

    end_time = time.time()

    # --- Checkpointing ---
    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss
        torch.save(model.state_dict(), 'transformer_best_model_epoch=10.pt')
        print(f"✔️ Epoch {epoch+1}: New best model saved (Val Loss: {avg_valid_loss:.3f})")

    print(f'Epoch: {epoch+1:02} | Time: {end_time - start_time:.2f}s')
    print(f'\tTrain Loss: {avg_train_loss:.3f} | Train PPL: {math.exp(avg_train_loss):7.3f}')
    print(f'\t Val. Loss: {avg_valid_loss:.3f} |  Val. PPL: {math.exp(avg_valid_loss):7.3f}')

Epoch 1/10:   0%|          | 0/907 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

✔️ Epoch 1: New best model saved (Val Loss: 3.257)
Epoch: 01 | Time: 56.64s
	Train Loss: 3.452 | Train PPL:  31.562
	 Val. Loss: 3.257 |  Val. PPL:  25.971


Epoch 2/10:   0%|          | 0/907 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

✔️ Epoch 9: New best model saved (Val Loss: 2.258)
Epoch: 09 | Time: 62.57s
	Train Loss: 2.146 | Train PPL:   8.547
	 Val. Loss: 2.258 |  Val. PPL:   9.567


Epoch 10/10:   0%|          | 0/907 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

✔️ Epoch 10: New best model saved (Val Loss: 2.191)
Epoch: 10 | Time: 62.06s
	Train Loss: 2.052 | Train PPL:   7.785
	 Val. Loss: 2.191 |  Val. PPL:   8.946


In [101]:
torch.cuda.empty_cache()

In [103]:
import math
import time
from tqdm.notebook import tqdm

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    val_pbar = tqdm(iterator, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in val_pbar:
            src = batch["en_ids"].to(device).transpose(0, 1)
            trg = batch["de_ids"].to(device).transpose(0, 1)

            output = model(src, trg[:, :-1])

            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    return epoch_loss / len(iterator)

# --- Hyperparameters & Early Stopping Setup ---
num_epochs = 50  # Increased since Early Stopping will handle the cutoff
best_valid_loss = float('inf')
patience = 7      # How many epochs to wait for improvement before stopping
counter = 0       # Tracks epochs without improvement

for epoch in range(num_epochs):
    start_time = time.time()

    # --- Training Phase ---
    model.train()
    train_loss = 0
    train_pbar = tqdm(train_data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in train_pbar:
        src = batch["en_ids"].to(device).transpose(0, 1)
        trg = batch["de_ids"].to(device).transpose(0, 1)

        optimizer.zero_grad()
        output = model(src, trg[:, :-1])

        output_dim = output.shape[-1]
        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    # --- Validation Phase ---
    avg_train_loss = train_loss / len(train_data_loader)
    avg_valid_loss = evaluate(model, valid_data_loader, criterion)

    end_time = time.time()

    # --- Checkpointing & Early Stopping Logic ---
    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss
        torch.save(model.state_dict(), 'transformer_best_model_epoch=50.pt')
        print(f"✔️ Epoch {epoch+1}: New best model saved! (Val Loss: {avg_valid_loss:.3f})")
        counter = 0  # Reset counter because we found a better model
    else:
        counter += 1
        print(f"⚠️ No improvement in Val Loss for {counter} epoch(s).")

    print(f'Epoch: {epoch+1:02} | Time: {end_time - start_time:.2f}s')
    print(f'\tTrain Loss: {avg_train_loss:.3f} | Train PPL: {math.exp(avg_train_loss):7.3f}')
    print(f'\t Val. Loss: {avg_valid_loss:.3f} |  Val. PPL: {math.exp(avg_valid_loss):7.3f}')

    if counter >= patience:
        print(f"🛑 Early stopping triggered. Training finished at Epoch {epoch+1}.")
        break

Epoch 1/50:   0%|          | 0/907 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

✔️ Epoch 1: New best model saved! (Val Loss: 1.876)
Epoch: 01 | Time: 64.43s
	Train Loss: 1.029 | Train PPL:   2.798
	 Val. Loss: 1.876 |  Val. PPL:   6.525


Epoch 2/50:   0%|          | 0/907 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

⚠️ No improvement in Val Loss for 6 epoch(s).
Epoch: 07 | Time: 60.27s
	Train Loss: 0.834 | Train PPL:   2.302
	 Val. Loss: 1.945 |  Val. PPL:   6.992


Epoch 8/50:   0%|          | 0/907 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

⚠️ No improvement in Val Loss for 7 epoch(s).
Epoch: 08 | Time: 63.43s
	Train Loss: 0.805 | Train PPL:   2.238
	 Val. Loss: 1.943 |  Val. PPL:   6.979
🛑 Early stopping triggered. Training finished at Epoch 8.


## Inference code

In [104]:
def translate_sentence(sentence, model, en_vocab, de_itos, device, max_len=50):
    model.eval()

    # 1. Tokenize and convert to indices
    tokens = [token.text.lower() for token in en_nlp.tokenizer(sentence)]
    tokens = ['<sos>'] + tokens + ['<eos>']
    src_indices = [en_vocab.get(token, en_vocab['<unk>']) for token in tokens]

    # 2. Convert to Tensor and move to device [Batch, Seq]
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)

    # 3. Start the decoder with just <sos>
    trg_indices = [de_vocab['<sos>']]

    for i in range(max_len):
        trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(src_tensor, trg_tensor)

        # Take the last token predicted
        preds = output.argmax(2)[:, -1].item()
        trg_indices.append(preds)

        # Stop if model predicts <eos>
        if preds == de_vocab['<eos>']:
            break

    # Convert indices back to words (skipping <sos> and <eos>)
    translated_tokens = [de_itos[idx] for idx in trg_indices if idx not in [de_vocab['<sos>'], de_vocab['<eos>']]]
    return " ".join(translated_tokens)

In [105]:
# List your model paths here
model_paths = [
    '/content/transformer_best_model_epoch=10.pt',
    '/content/transformer_best_model_epoch=50.pt',
    '/content/transformer_best_model.pt'
]

test_sentence = "A small dog runs through the green grass."

print(f"Source: {test_sentence}\n" + "-"*30)

for path in model_paths:
    try:
        # Load the weights into the existing model architecture
        model.load_state_dict(torch.load(path))
        translation = translate_sentence(test_sentence, model, en_vocab, de_itos, device)
        print(f"Model [{path}]:\n -> {translation}\n")
    except FileNotFoundError:
        print(f"Skipping: {path} not found.")

Source: A small dog runs through the green grass.
------------------------------
Model [/content/transformer_best_model_epoch=10.pt]:
 -> ein kleiner hund rennt durch das grünen gras .

Model [/content/transformer_best_model_epoch=50.pt]:
 -> ein kleiner hund rennt durch das grüne gras .

Model [/content/transformer_best_model.pt]:
 -> ein junger hund springt auf dem boden .



In [106]:
import pickle

# Define what we need to save
deployment_package = {
    "en_vocab": en_vocab,
    "de_vocab": de_vocab,
    "de_itos": de_itos,
    "d_k": d_k,
    "heads": heads,
    "head_dim": head_dim,
    "max_length": max_length,
    "pad_index": pad_index
}

# Save metadata to a file
with open('transformer_metadata.pkl', 'wb') as f:
    pickle.dump(deployment_package, f)

print("✅ Metadata saved to transformer_metadata.pkl")

✅ Metadata saved to transformer_metadata.pkl
